In [2]:
import requests
import os

In [3]:
API_KEY = os.getenv("DGIS_API_KEY")

In [4]:
url = "https://catalog.api.2gis.com/3.0/items"

params = {
    "q": "кофейня",
    "point": "76.945,43.238",
    "radius": 1000,
    "type": "branch",
    "page_size": 10,
    "key": API_KEY
}

response = requests.get(url, params=params, timeout=15)
print(response.status_code)


200


In [5]:
from dotenv import load_dotenv
import os

load_dotenv('/Users/david/Desktop/business-location-ai/business-location-ai/.env', override=True)

API_KEY = os.getenv('DGIS_API_KEY')
print("Ключ загружен:", bool(API_KEY))

Ключ загружен: True


In [6]:
response = requests.get(url, params=params, timeout=15)
data = response.json()

print("HTTP:", response.status_code)
print("API code:", data.get("meta", {}).get("code"))
print("Количество организаций:", len(data.get("result", {}).get("items", [])))

HTTP: 200
API code: 200
Количество организаций: 10


In [7]:
items = data['result']['items']

print("Количество:", len(items))
print("Поля организации:", items[0].keys())

Количество: 10
Поля организации: dict_keys(['address_comment', 'address_name', 'id', 'name', 'type'])


In [8]:
import json

print(json.dumps(items[0], indent=2, ensure_ascii=False))

{
  "address_comment": "1 этаж",
  "address_name": "проспект Абая, 17",
  "id": "70000001052408859",
  "name": "Six Coffee+Wine, кофейня",
  "type": "branch"
}


In [9]:
params["fields"] = "items.point,items.reviews"

response = requests.get(url, params=params, timeout=15)
data = response.json()

print("API code:", data["meta"]["code"])

if "result" in data and data["result"].get("items"):
    print(json.dumps(
        data["result"]["items"][0],
        indent=2,
        ensure_ascii=False
    ))

API code: 200
{
  "address_comment": "1 этаж",
  "address_name": "проспект Абая, 17",
  "id": "70000001052408859",
  "name": "Six Coffee+Wine, кофейня",
  "point": {
    "lat": 43.243281,
    "lon": 76.953818
  },
  "reviews": {
    "general_rating": 4.9,
    "general_review_count": 5641,
    "general_review_count_with_stars": 9642,
    "is_reviewable": false,
    "is_reviewable_on_flamp": true,
    "items": [
      {
        "is_reviewable": true,
        "tag": "2gis_reviews"
      }
    ],
    "org_rating": 4.7,
    "org_review_count": 6422,
    "org_review_count_with_stars": 11517
  },
  "type": "branch"
}


In [10]:
import geopandas as gpd
from shapely.geometry import Point

items = data['result']['items']

coffee_points = []

for item in items:
    if 'point' in item:
        longitude = item['point']['lon']
        latitude = item['point']['lat']

        coffee_points.append(Point(longitude, latitude))


print(len(coffee_points))


10


In [11]:
coffee_gdf = gpd.GeoDataFrame(
    geometry=coffee_points,
    crs="EPSG:4326"
)



In [12]:
coffee_gdf_m = coffee_gdf.to_crs("EPSG:32643")


coffee_gdf_m.head()

,geometry
0,POINT (658624.682 4789684.598)
1,POINT (657718.565 4790004.611)
2,POINT (657378.892 4788938.449)
3,POINT (657118.299 4789137.312)
4,POINT (658115.917 4789862.515)


In [19]:
import geopandas as gpd
from shapely.geometry import Point

items = data['result']['items']

coffee_records = []

for item in items:
    if 'point' in item:
        longitude = item['point']['lon']
        latitude = item['point']['lat']

        coffee_records.append({
            'id': item['id'],
            'name': item['name'],
            'geometry': Point(longitude, latitude),
            'general_rating' : item["reviews"]["general_rating"],
            'general_review_count' : item["reviews"]["general_review_count"]
        })

print(len(coffee_records))


10


In [20]:
coffee_gdf = gpd.GeoDataFrame(
    coffee_records,
    geometry="geometry",
    crs="EPSG:4326"
).to_crs("EPSG:32643")

coffee_gdf.head()

,id,name,geometry,general_rating,general_review_count
0,70000001052408859,"Six Coffee+Wine, кофейня",POINT (658624.682 4789684.598),4.9,5641
1,70000001104437840,"BAL, кофейня",POINT (657718.565 4790004.611),4.9,2289
2,70000001084456166,"Arabica city cafe, городское кафе",POINT (657378.892 4788938.449),4.8,1137
3,70000001101132415,"Bellocake_kz, кондитерская",POINT (657118.299 4789137.312),4.7,969
4,70000001105830997,"Skyberry, кофейня",POINT (658115.917 4789862.515),4.9,472


In [21]:
print("Количество кофеен:", len(coffee_gdf))
print("Уникальных ID:", coffee_gdf["id"].nunique())
print("Пропущенных координат:", coffee_gdf.geometry.isna().sum())
print("CRS:", coffee_gdf.crs)

Количество кофеен: 10
Уникальных ID: 10
Пропущенных координат: 0
CRS: EPSG:32643


In [ ]:
params['page'] = 2


response = requests.get(url, params=params, timeout=15)
data_page_2 = response.json()

print("API code:", data_page_2["meta"]["code"])
print("Количество:", len(data_page_2.get("result", {}).get("items", [])))

API code: 200
Количество: 10


In [24]:
ids_page_1 = {item["id"] for item in data["result"]["items"]}
ids_page_2 = {item["id"] for item in data_page_2["result"]["items"]}

print("Общих ID:", len(ids_page_1 & ids_page_2))
print("Всего уникальных:", len(ids_page_1 | ids_page_2))

Общих ID: 1
Всего уникальных: 19


In [ ]:
for i in range(1,6):
    params['page'] = i

    response = requests.get(url, params=params, timeout=15)
    data_page = response.json()

    print("Страница:", i)
    print("API code:", data_page["meta"]["code"])


    if data_page['meta']['data'] !=200:
        break

    page_items = data_page.get()

